# Choosing a Finite-Difference Strategy for Gradient Checking and AD-via-FD

In this tutorial, you will learn how to:

1. **See why a single global `eps` fails** when inputs span several orders of magnitude or the function is noisy
2. **Compare four ways to pick the step size**: fixed, per-input, input-normalized, and best-of-sweep
3. **Tell a wrong gradient from a noisy finite difference** in `check-gradients`, including with a Taylor-remainder test
4. **Judge whether FD gradients served by AD-via-FD endpoints are good enough** to drive an optimizer
5. **Pick a default strategy and tolerance** for each use

## Context

Finite differences (FD) play two roles in Tesseract:

- **Referee.** `tesseract-runtime check-gradients` compares a Tesseract's own `jacobian` / `jacobian_vector_product` / `vector_jacobian_product` against FD. The question is whether a correct gradient passes and a slightly wrong one fails.
- **Provider.** The helpers in `tesseract_core.runtime.experimental` (`finite_difference_jacobian`, `finite_difference_jvp`, `finite_difference_vjp`) *are* the gradient when a Tesseract only implements `apply`. The question is whether that gradient is accurate enough to optimize with.

Both roles depend on the same choice: the step size `eps`. A badly chosen `eps` makes correct gradients fail the check, and makes served gradients too inaccurate to optimize with.

<!-- TODO: 2-3 sentence motivating anecdote (#706): correct gradients, 400/400 failures, caused only by eps on an input of magnitude ~2e4. -->

This notebook compares the strategies discussed in [#749](https://github.com/pasteurlabs/tesseract-core/issues/749):

| Strategy | Referee | Provider | Built into Tesseract today |
| --- | --- | --- | --- |
| Fixed global `eps` | yes | yes | yes (default `1e-4`) |
| Per-input `eps` | yes | yes | yes (mapping of path → `eps`) |
| Input-normalized `eps` | yes | yes | no, implemented here |
| Best-of-sweep `eps` | yes | yes | no, implemented here |
| Taylor-remainder order test | yes | no | no, implemented here |

> **Design notes (remove before merge).** Open decisions to settle with the team before implementation:
>
> - **D1: Where do the new strategies live?** (a) Prototypes in this notebook, labelled as such, with results feeding follow-up PRs; or (b) land sweep and normalization upstream first. Default: (a).
> - **D2: Which real Tesseract for Step 6?** Default: `bayesian-inference/lorenz_tesseract_finitediff` (NumPy, float32, FD endpoints at `eps=1e-5`) against its analytic JAX twin `data-assimilation/lorenz_tesseract`. mosaic solvers are what #749 asks for, but are likely not public.
> - **D3: How is the ~5% bug planted?** Default: scale one Jacobian term by 1.05, mimicking pasteurlabs/mosaic#158. The same bug is used for every strategy.
> - **D4: CI budget.** CI executes this notebook with a 60 min limit, including image builds. Default: a reduced seed/noise grid, total runtime < 10 min.
> - **Adjoint identity (JVP against VJP):** in or out? Default: one line in Takeaways, no experiment. Most target solvers expose only a VJP.
> - **One-hot vs dense random probes** (#739): optional Step 4b, only if time allows.

In [ ]:
%pip install -r requirements.txt -q

## Step 1: A toy function with known pathologies

We start with a function whose exact gradient we know, so we can measure FD error directly. It is built to contain the failure modes seen in practice:

- **Two inputs about four orders of magnitude apart**, like a drag coefficient `alpha ~ 2e4` next to a temperature `T ~ 1` (#706).
- **A noise knob** standing in for solver convergence tolerance, so the FD noise floor is not machine epsilon.
- **Planted gradient variants**: the correct gradient, a ~5% wrong one (one term scaled), and a blatant one (sign flip) as a sanity check.

We wrap it as a Tesseract API module so the same object can go through `check_gradients` and the experimental FD helpers.

In [ ]:
# TODO: define the toy f(alpha, T; noise_level) and its exact gradient.
# TODO: gradient variants: "correct", "5pct_wrong", "sign_flip".
# TODO: wrap as a Tesseract API module (InputSchema/OutputSchema, apply, vjp, jvp, jacobian)
#       selectable by variant, so check_gradients and the FD helpers can both use it.
# TODO: evaluation counter (as in bayesian-inference/_test_e2e.py) to report cost per strategy.

## Step 2: Why a single `eps` fails

The FD error has two parts: truncation error, which shrinks as `h` shrinks, and noise amplification, which grows as `h` shrinks. Plotting error against `h` gives a V shape; the bottom of the V is the best achievable step.

Because `alpha` and `T` have very different scales, their V's bottom out at very different `h`. No single `eps` sits at the bottom of both.

In [ ]:
# TODO: for each input and noise level, sweep h over ~1e-12..1e0 and plot
#       |FD - exact| / |exact| for central differences (forward shown faintly for reference).
# TODO: mark the default eps=1e-4 on the plot.
# Output: one figure, two panels (alpha, T), one line per noise level.

## Step 3: Four ways to choose `eps`

We compare the strategies on the toy, against the exact gradient:

1. **Fixed global `eps`**: the current default.
2. **Per-input `eps`**: one step per input path, chosen by hand (supported via a mapping since #713).
3. **Input-normalized `eps`**: scale each step by the input's magnitude so one relative `eps` works for all inputs (#516, #706).
4. **Best-of-sweep `eps`**: try a coarse set such as `[1e-1, 1e-2, 1e-3, 1e-4]` and keep the most self-consistent estimate (as mosaic does, #740).

For each we report the FD error and the number of `apply` calls. The best error achievable here sets the tightest `rtol` it makes sense to demand in Step 4.

In [ ]:
# TODO: implement normalized and best-of-sweep strategies (prototype, see design note D1).
# TODO: define the selection rule for best-of-sweep (e.g. smallest change between neighbouring eps).

In [ ]:
# TODO: evaluate each strategy at each noise level; collect rel. error vs exact and apply-call count.
# Output: table  strategy | noise | rel err (alpha) | rel err (T) | apply calls

## Step 4: The referee — does `check-gradients` catch a wrong gradient?

Now we pretend the exact gradient is unknown. For each strategy, noise level, and seed, we run `check_gradients` on the correct and on the 5%-wrong variant and record:

- whether the correct gradient passes (no false alarm),
- whether the 5%-wrong gradient is caught,
- the loosest and tightest `rtol` that separate the two.

We also run the **Taylor-remainder test**: for a correct gradient, the remainder `|f(x+hd) - f(x) - h <g, d>|` shrinks like `h²`; for a wrong one, like `h`. It needs only a VJP. As #740 found, the slope must be read over coarse `h`, because fine `h` is dominated by noise.

In [ ]:
# TODO: grid over strategy x noise x seed x variant; call check_gradients with rtol in a small sweep.
# TODO: Taylor-remainder test over a coarse h window; report fitted slope per seed.
# Output: table  strategy | noise | correct pass (k/N) | 5%-wrong caught (k/N) | tightest working rtol

### Is the default `rtol=0.1` too loose?

<!-- TODO: one paragraph + small plot: catch rate of the 5% bug vs rtol, per strategy. -->

In [ ]:
# TODO: catch rate of "5pct_wrong" and pass rate of "correct" as a function of rtol.

## Step 5: The provider — are AD-via-FD gradients good enough to optimize?

Here FD is not checking a gradient, it *is* the gradient. We build AD-via-FD endpoints for the toy using each `eps` strategy and ask two questions:

- How far is the served gradient from the exact one?
- Does a short gradient-descent run converge to the known optimum?

In [ ]:
# TODO: AD-via-FD endpoints (finite_difference_vjp etc.) per strategy on the toy.
# TODO: short gradient descent from a fixed start; record final objective and iterations.
# Output: table  strategy | rel err vs exact | optimizer converges | apply calls

## Step 6: A real Tesseract

We repeat Steps 4 and 5 on a real simulator: the Lorenz 96 Tesseract from the [Bayesian inference demo](bayesian-inference.ipynb). Its `lorenz_tesseract_finitediff` variant serves FD gradients (NumPy, float32), and the analytic JAX version from the [data assimilation demo](data-assimilation.ipynb) gives us a reference gradient.

Float32 raises the noise floor to about `1e-7`, which moves the bottom of the V and changes which `eps` works.

In [ ]:
%%bash
# Render build status statically instead of an animated spinner,
# which streams as noise in captured notebook output.
export TERM=dumb
# TODO: tesseract build ../bayesian-inference/lorenz_tesseract_finitediff/
# TODO: tesseract build ../data-assimilation/lorenz_tesseract/

In [ ]:
# TODO: serve both Tesseracts.
# TODO: referee: check_gradients on the analytic Tesseract with each strategy (+ a planted 5% bug via a wrapper).
# TODO: provider: rel. error of FD gradients vs analytic, per strategy; short optimization recovering F.

## Step 7: Comparison

<!-- TODO: the two headline tables from #749, one for each role, filled from Steps 4-6. -->

In [ ]:
# TODO: assemble summary tables:
#   [check-gradients]  strategy | noise | correct pass | 5%-wrong caught
#   [ad-via-fd]        strategy | rel err vs reference | optimizer converges

## Cleanup

In [ ]:
# TODO: tear down served Tesseracts.

## Takeaways

<!-- TODO: fill from results. Each takeaway should name a strategy and a number. -->

1. **Referee:** <!-- recommended default strategy for check-gradients -->
2. **Provider:** <!-- recommended default strategy for AD-via-FD endpoints -->
3. **Tolerance:** <!-- verdict on rtol=0.1 -->
4. **When to use the Taylor test:** <!-- the band where it beats FD, if any -->
5. **Cost:** <!-- apply calls per strategy -->

### What's next

- **Random probes instead of one-hot sampling** in `check-gradients` ([#739](https://github.com/pasteurlabs/tesseract-core/pull/739)).
- **Stochastic objectives:** SPSA and common random numbers for noisy simulators.
- **Adjoint identity:** check JVP against VJP without FD when a Tesseract exposes both.
- **Explore other demos** that rely on FD gradients, such as the [Bayesian inference demo](bayesian-inference.ipynb).

Questions? Feedback? Please reach out through the [Tesseract Community Forum](https://si-tesseract.discourse.group/).